# Распределения случайной величины

In [2]:
import numpy as np
from scipy import stats
import plotly.graph_objects as go

# Размер выборки для визуализации
N = 5000
seed = 42


def plot_continuous(dist, samples, xs, title, x_title="x"):
    """Гистограмма выборки + теоретическая PDF (для непрерывных распределений)."""
    fig = go.Figure()
    fig.add_trace(
        go.Histogram(
            x=samples,
            histnorm="probability density",
            name="Эмпирическая плотность",
            opacity=0.35,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=xs,
            y=dist.pdf(xs),
            mode="lines",
            name="Теоретическая PDF",
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title=x_title,
        yaxis_title="Плотность",
        template="plotly_white",
        height=450,
    )
    return fig


def plot_discrete(dist, samples, ks, title, x_title="k"):
    """PMF (теория) + эмпирические вероятности из выборки (для дискретных)."""
    pmf = dist.pmf(ks)
    emp = np.array([(samples == k).mean() for k in ks])

    fig = go.Figure()
    fig.add_trace(go.Bar(x=ks, y=pmf, name="Теоретическая PMF", opacity=0.6))
    fig.add_trace(go.Bar(x=ks, y=emp, name="Эмпирическая частота", opacity=0.35))
    fig.update_layout(
        title=title,
        xaxis_title=x_title,
        yaxis_title="Вероятность",
        template="plotly_white",
        height=450,
    )
    return fig


## Нормальное распределение случайной величины

Симметричное «колокол»-образное распределение

Описывается:
- μ — среднее
- σ — стандартное отклонение

Где встречается:

- ошибки измерений
- A/B тесты (при больших выборках)
- средние значения метрик
- шум в данных

$$f(x)=\frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

Это является основным распределением, т. к. по центральной предельной теореме **сумма или среднее большого числа случайных величин → стремится к нормальному распределению**

In [ ]:
# 1) Нормальное распределение
mu, sigma = 0, 1
normal = stats.norm(loc=mu, scale=sigma)
normal_samples = normal.rvs(size=N, random_state=seed)
xs_normal = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 400)
plot_continuous(normal, normal_samples, xs_normal, "Нормальное распределение (Normal)")

: 

## Логнормальное распределение

Если логарифм величины распределён нормально → сама величина логнормальна.

**Формально:**
Если
$$
\ln(X) \sim N(\mu, \sigma^2)
$$
то X — логнормальная

**Свойства:**

- только положительные значения
* сильный правый хвост (skewed right)

**Где используется:**

* доходы пользователей
* время до события (lifetime)
* размеры файлов / запросов
* цены


-> если данные «перекошены вправо» → попробуй логарифм


In [4]:
# 2) Логнормальное распределение: ln(X) ~ N(mu_ln, sigma_ln^2)
mu_ln, sigma_ln = 0.0, 0.6
lognormal = stats.lognorm(s=sigma_ln, scale=np.exp(mu_ln))
lognormal_samples = lognormal.rvs(size=N, random_state=seed)
lo, hi = np.percentile(lognormal_samples, [0.1, 99.9])
xs_lognormal = np.linspace(max(1e-9, lo), hi, 400)
plot_continuous(
    lognormal,
    lognormal_samples,
    xs_lognormal,
    "Логнормальное распределение (Lognormal)",
)


## Биномиальное распределение

Дискретное распределение для количества успехов.

$$P(X=k)=\binom{n}{k} p^k (1-p)^{n-k}$$

**Параметры:**

* n — число испытаний
* p — вероятность успеха

**Примеры:**

* клики (кликнул / не кликнул)
* конверсии
* успешные транзакции


In [17]:
# 3) Биномиальное распределение
n_trials, p_binom = 3, 0.5
binom = stats.binom(n=n_trials, p=p_binom)
binom_samples = binom.rvs(size=N, random_state=seed)
ks_binom = np.arange(0, n_trials + 1)
plot_discrete(
    binom,
    binom_samples,
    ks_binom,
    "Биномиальное распределение (Binomial)",
)

## Связь между биноминальным и нормальным распределением

Это ключевой момент.

Когда:

* n большое (обычно n > 30)
* p не слишком близко к 0 или 1

-> биномиальное распределение можно приблизить нормальным:

$$
X \sim Bin(n, p) \approx N(\mu = np,\ \sigma^2 = np(1-p))
$$

**Почему это работает:**
Опять же из-за Центральная предельная теорема

Интуиция:

* биномиальное = сумма n независимых Бернулли
* сумма → становится нормальной

**Практика:**

* z-тесты для долей
* доверительные интервалы
* A/B тесты

## Распределение Бернулли

Частный случай биномиального:

* n = 1

$$
X \sim Bern(p)
$$

$$p_0+p_1 = 1$$

**Значения:**

* 1 — успех
* 0 — неудача

-> основа для:

* биномиального
* логистической регрессии

In [ ]:
# 4) Распределение Бернулли (частный случай биномиального)
p_bern = 0.9
bern = stats.bernoulli(p=p_bern)
bern_samples = bern.rvs(size=N)
ks_bern = np.array([0, 1])
plot_discrete(bern, bern_samples, ks_bern, "Распределение Бернулли (Bernoulli)")

In [15]:
bern_samples

array([1, 1, 1, ..., 1, 0, 1], shape=(5000,))

## Пуассоновское распределение

Для счёта редких событий за время.

$$
P(X=k)=\frac{\lambda^k e^{-\lambda}}{k!}
$$

**Где используется:**

* число ошибок в системе
* количество заказов в минуту
* события в потоке

**Связь:**
биномиальное → пуассон:

* n большое
* p маленькое
* np = λ

In [8]:
# 5) Пуассоновское распределение
lambda_poiss = 1
poisson = stats.poisson(mu=lambda_poiss)
poisson_samples = poisson.rvs(size=N, random_state=seed)
# берём поддержку вокруг ожидаемого значения
k_max = int(lambda_poiss + 5 * np.sqrt(lambda_poiss))
ks_poisson = np.arange(0, k_max + 1)
plot_discrete(
    poisson,
    poisson_samples,
    ks_poisson,
    "Пуассоновское распределение (Poisson)",
)

## Экспоненциальное распределение

Время между событиями в пуассоновском процессе.

$$
f(x)=\lambda e^{-\lambda x}
$$

**Смысл:**

* сколько ждать до следующего события

**Примеры:**

* время до следующего клика
* время до отказа сервера


In [9]:
# 6) Экспоненциальное распределение: f(x)=lambda*exp(-lambda x), x>=0
rate = 1  # lambda
expon = stats.expon(scale=1.0 / rate)
expon_samples = expon.rvs(size=N, random_state=seed)
lo, hi = np.percentile(expon_samples, [0.1, 99.9])
xs_expon = np.linspace(max(0.0, lo), hi, 400)
plot_continuous(expon, expon_samples, xs_expon, "Экспоненциальное распределение (Exponential)")

## Как всё связано (очень важно)

Вот реальная карта:

* Bernoulli → Binomial
* Binomial → Normal (при больших n)
* Binomial → Poisson (при малом p)
* Poisson → Exponential (интервалы времени)
* Любая сумма → Normal (через Центральная предельная теорема)